In [1]:
# import sys
# sys.path.insert(0, 'utils.py')

In [1]:
# /!\ Before running the lab make sure every additional libraries is installed 
# Import local libraries
# ***change path in utils.py according to the scenario1 folder location!***
from utils import *
import time

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


INFO - 2026-10-20 17:27:28,573 - utils - NumExpr defaulting to 4 threads.
/user/rcoelhopae2025/.local/lib/python3.8/site-packages/pandas/core/computation/expressions.py:20: UserWarning: Pandas requires version '2.7.3' or newer of 'numexpr' (version '2.7.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


# 1 Reading of one LiDAR scan

In [2]:
n_frame = 0
actor = 'ego_vehicle'

points = get_point_cloud(n_frame, actor)

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)

flag_display = True

In [3]:
o3d.visualization.draw_geometries([pcd])

# 1.1 Voxel Down sampling

To reduce the number of points and accelerate the computation use the function _voxel_down_sample()_ of open3D.

In [4]:
downpcd = pcd.voxel_down_sample(voxel_size=0.5) # ToDo (Replace this line) 

if flag_display:
    o3d.visualization.draw_geometries([downpcd])

# 1.2 Normals estimation
We would like to detect obstacles, to do that the first step is to estimate normals to obtain the local orientation of the point cloud.

Use the function _estimate_normals()_ of open3D to estimate this normals.

In [5]:
#ToDo estimate the normals
normal_estimat = downpcd.estimate_normals() 

if flag_display:
    o3d.visualization.draw_geometries([downpcd], point_show_normal=True)

In [6]:
downpcd.colors = o3d.utility.Vector3dVector(np.abs(np.array(downpcd.normals)))
# the color represents the direction of the normal
if flag_display:
    o3d.visualization.draw_geometries([downpcd])

# 1.3 Ground and objects segmentation

Based on 
- the height of each point, 
- the local orientation of the points cloud arround this point
- the variation of this orientation,

Filter the ground to obtain a new points cloud whith only the ground points and another one with only the objects on top of this ground.

For both of these new points clouds you can also filter the points corresponding to the roof of the ego vehicle.

In [7]:
def ground_filtering(pcd):
    plane_model,pcd_ground = pcd.segment_plane(distance_threshold=0.5,ransac_n=3,num_iterations=1000) # ToDo (Replace this line) 
    return pcd_ground

pcd_ground_i = ground_filtering(downpcd) # index of the ground points according to threshold

pcd_ground = downpcd.select_by_index(pcd_ground_i,False) # take only the ground voxels

if flag_display:
    o3d.visualization.draw_geometries([pcd_ground])

In [8]:
def objects_filtering(pcd):
    plane_model,pcd_objects = pcd.segment_plane(distance_threshold=0.5,ransac_n=3,num_iterations=1000) # ToDo (Replace this line) 
    return pcd_objects

pcd_objects_i = objects_filtering(downpcd)
pcd_objects = downpcd.select_by_index(pcd_ground_i,True) # take only the object voxels (inverting indeces)

if flag_display:
    o3d.visualization.draw_geometries([pcd_objects])

# 1.4 Clustering

Now that the points of the objects are not connected to the ground anymore we can regroup all the points of the same object on one cluster.

To do that, use the function _cluster_dbscan()_ of open3D to obtain the cluster id for each point.

In [13]:
def clustering(pcd_objects):
    
    cluster_ids = pcd_objects.cluster_dbscan(2,20)
    cluster_ids = np.asarray(cluster_ids) # transforming it in np array
    cluster_nb = cluster_ids.max()  #cluster_ids.max()
    
    colors = plt.get_cmap("tab20")(cluster_ids / (cluster_nb if cluster_nb > 0 else 1))
    colors[cluster_ids < 0] = 0
    pcd_objects.colors = o3d.utility.Vector3dVector(colors[:, :3])
    return pcd_objects, cluster_ids

pcd_objects, cluster_ids = clustering(pcd_objects)

print("The points cloud has", cluster_ids.max() + 1, "clusters")

if flag_display:
    o3d.visualization.draw_geometries([pcd_objects])

The points cloud has 12 clusters


# 1.5 Bounding Boxes estimation
Estimate for each of these clusters its bounding box. You can use the function _get_axis_aligned_bounding_box()_ or _get_oriented_bounding_box()_ of open3D.

What are the limits of these functions ?

You can keep only the small bounding boxes correponding to the small objects (such as vehicles or pedestrians and not buildings)

In [14]:
def cluster_boxes(pcd_objects, cluster_ids):
    cluster_n = cluster_ids.max() + 1 # total number of clusters
    boxes = []
    
    for i in range(cluster_n):
        pcd_obj = pcd_objects.select_by_index(np.where(cluster_ids == i)[0])
        pdc_obj_box = pcd_obj.get_axis_aligned_bounding_box()
        
        # Bounding boxes
        boxes.append(pdc_obj_box)
    return boxes

boxes = cluster_boxes(pcd_objects, cluster_ids)

obj_to_display = boxes
obj_to_display.append(pcd_objects)

if flag_display:
    o3d.visualization.draw_geometries(obj_to_display)

In [12]:
np.asarray(pcd_objects.points)

array([[  1.76223511,   0.67969858,   0.85136502],
       [ -0.94452308,  -0.09873555,   1.38618186],
       [  1.79783162,  -0.1158928 ,   0.92511036],
       ...,
       [ -8.1123353 , -54.64859466,   7.87450348],
       [ -6.25120732, -20.67548963,   1.33373073],
       [  2.64645862,   8.68164477,   1.89084135]])

# 2 Objects detection and tracking on LiDAR scans

Applied the previous functions on the different scan to estimate the bounding boxes.

### 2.1 Bounding boxes association (bonus)

Create a function to associate the bounding boxes of the current time to the one of the previous time based the shortest distances of there centers.

### 2.2 Bounding boxes orientation estimation (bonus)
Based on these association estimate the orientation of there motion to correct the orientation of the bounding boxes.

### 2.3 Points cloud mapping (bonus)
Build a map by accumulating the different points cloud and removing the moving objects (keep the cluster id and add a new one for the ground points)

### 2.4 Iterative Closest Point tracking (bonus)
To better associate the clusters you can also use the function _icp()_ of open3D on each cluster

In [23]:
# Initialize Open3D Visualizer
if flag_display:
    vis = o3d.visualization.Visualizer()
    vis.create_window()
    

for n_frame in range(nb_frames):
    
    # Read point cloud
    points = get_point_cloud(n_frame, actor)
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)
    
    # Voxel down sampling
    downpcd = pcd.voxel_down_sample(voxel_size=0.5) # ToDo (Replace this line)
    
    # Transform points cloud from the LiDAR to the world frame
    tf = get_tf_lidar2world(actor, n_frame)
    
    # ToDo apply this transformation to the points cloud
    #downpcd_t = apply_tf(tf,np.asarray(downpcd.points))
    downpcd_t = downpcd.transform(tf)
    
    # Estimate normals
    #ToDo estimate the normals
    normal_estimat = downpcd_t.estimate_normals() 
    downpcd_t.colors = o3d.utility.Vector3dVector(np.abs(np.array(downpcd_t.normals)))
    
    # Filter ground points
    pcd_ground = downpcd_t.select_by_index(ground_filtering(downpcd_t),False)
    
    # Filter objects points
    pcd_objects = downpcd_t.select_by_index(objects_filtering(downpcd_t),True)
    
    print(pcd_objects)
    # Clustering
    pcd_objects, cluster_ids = clustering(pcd_objects)

    if flag_display:
        # Clear previous geometries and add new point cloud to visualizer
        vis.clear_geometries()
        vis.add_geometry(pcd_objects)
        
        # Update the visualizer to show the new point cloud
        vis.poll_events()
        vis.update_renderer()

        # Wait before updating the points cloud (adjust time as needed)
        time.sleep(0.1)

if flag_display:
    # Close the visualizer
    vis.destroy_window()

PointCloud with 2942 points.
PointCloud with 2946 points.
PointCloud with 2963 points.
PointCloud with 3076 points.
PointCloud with 2882 points.
PointCloud with 2924 points.
PointCloud with 2791 points.
PointCloud with 2893 points.
PointCloud with 2700 points.
PointCloud with 2733 points.
PointCloud with 2680 points.
PointCloud with 2801 points.
PointCloud with 2755 points.
PointCloud with 2723 points.
PointCloud with 2622 points.
PointCloud with 2671 points.
PointCloud with 2674 points.
PointCloud with 2689 points.
PointCloud with 2773 points.
PointCloud with 2847 points.
PointCloud with 2777 points.
PointCloud with 2789 points.
PointCloud with 2751 points.
PointCloud with 2641 points.
PointCloud with 2752 points.
PointCloud with 2840 points.
PointCloud with 2714 points.
PointCloud with 2865 points.
PointCloud with 2802 points.
PointCloud with 2973 points.
PointCloud with 2883 points.
PointCloud with 2808 points.
PointCloud with 2910 points.
PointCloud with 2951 points.
PointCloud wit